# Verificación reproducible del Taller 3

**Autores:** Jairo Serrano y Edwin Yunis.

Cuaderno complementario del dashboard y Jupyter Book. Utiliza únicamente la matriz del Excel original exportada a CSV. Las secciones 5–8 requieren NumPy y SciPy (scipy.stats.spearmanr). Ejecútalo en Jupyter desde la carpeta `book/` del repositorio.

## 1. Importación y validación de los datos

Se comprueba la dimensión 12×12 y que la matriz contiene 43 intereses.

In [ ]:
import csv
from pathlib import Path
from itertools import combinations

ruta = Path('../data/matriz_intereses.csv')
if not ruta.exists():
    ruta = Path('data/matriz_intereses.csv')
with ruta.open(encoding='utf-8-sig', newline='') as f:
    filas = list(csv.reader(f))
tematicas = filas[0][1:]
sectores = [r[0] for r in filas[1:]]
A = [[int(v) for v in r[1:]] for r in filas[1:]]
assert len(A) == 12 and all(len(r) == 12 for r in A)
assert sum(map(sum, A)) == 43
print('Sectores:', len(sectores), 'Temáticas:', len(tematicas), 'Intereses:', sum(map(sum, A)))

## 2. Algoritmo de ordenamiento binario (ROC)

Pesos descendentes de potencias de dos y alternancia de ordenamientos de filas y columnas hasta lograr estabilidad.

In [ ]:
def reorder_roc(matriz):
    m = [row[:] for row in matriz]
    row_ids = list(range(len(m)))
    col_ids = list(range(len(m[0])))
    history = []
    for step in range(100):
        row_scores = [sum(v * 2**(len(m[0])-j-1) for j,v in enumerate(row)) for row in m]
        row_order = sorted(range(len(m)),key=lambda i:-row_scores[i])
        m = [m[i] for i in row_order]
        new_row_ids = [row_ids[i] for i in row_order]
        col_scores = [sum(m[i][j]*2**(len(m)-i-1) for i in range(len(m))) for j in range(len(m[0]))]
        col_order = sorted(range(len(m[0])),key=lambda j:-col_scores[j])
        m = [[row[j] for j in col_order] for row in m]
        new_col_ids = [col_ids[j] for j in col_order]
        history.append((new_row_ids[:],new_col_ids[:]))
        if new_row_ids == row_ids and new_col_ids == col_ids: break
        row_ids,col_ids=new_row_ids,new_col_ids
    return m,row_ids,col_ids,history

roc_matrix, row_order, col_order, history = reorder_roc(A)
print('Iteraciones:', len(history))
print('Orden final de temáticas:')
for j in col_order: print('-', tematicas[j])

## 3. Evaluación ROC y prueba exhaustiva de optimización

Se enumeran las 15.400 particiones de doce temáticas en cuatro grupos no etiquetados de tres. La cota inferior por sector verifica el óptimo.

In [ ]:
def loss_for_groups(groups):
    return sum(sum(max(0,sum(row[j] for j in group)-1) for group in groups) for row in A)

roc_groups = [col_order[k:k+3] for k in range(0,12,3)]
print('Pérdidas ROC:', loss_for_groups(roc_groups))

# Enumeración sin repetir permutaciones de franjas ni auditorios.
counter = 0
best = float('inf')
best_groups = []
def search(remaining, groups):
    global counter, best, best_groups
    if not remaining:
        counter += 1
        loss = loss_for_groups(groups)
        if loss < best:
            best = loss
            best_groups = [tuple(tuple(x) for x in groups)]
        elif loss == best:
            best_groups.append(tuple(tuple(x) for x in groups))
        return
    first = min(remaining)
    for pair in combinations([v for v in remaining if v!=first],2):
        group = (first,)+pair
        search([v for v in remaining if v not in group], groups+[group])

search(list(range(12)),[])
print('Particiones estudiadas:',counter)
print('Pérdidas óptimas:',best)
print('Número de particiones óptimas:',len(best_groups))
assert counter == 15400 and best == 7
lower_bound = sum(max(0,sum(row)-4) for row in A)
print('Cota inferior teórica:',lower_bound)
assert lower_bound == best

## 4. Programación recomendada

Un representante sectorial puede asistir a una conferencia por franja; 7 pérdidas es el mínimo global.

In [ ]:
optimal = [[2,9,10],[0,3,7],[5,8,11],[1,4,6]]
hours = ['08:00–09:45','10:00–11:45','14:00–15:45','16:00–17:45']
assert loss_for_groups(optimal) == 7
print('Programa optimizado:')
for hour,group in zip(hours,optimal):
    print(hour, ' | '.join(tematicas[j] for j in group))
print('Aprovechamiento:',43-loss_for_groups(optimal),'de 43')

## Conclusión

El procedimiento ROC ofrece una organización por afinidad, pero la optimización reduce las oportunidades perdidas de 18 a 7, es decir, **61,11 %**.

El análisis es reproducible a partir de los datos del taller académico.

## 5. Correlación de Spearman entre sectores

Se calcula Spearman directamente a partir de los 12 vectores binarios de las 12 temáticas. Los empates se corrigen mediante rangos promedio. Para vectores binarios no constantes, Spearman coincide con el coeficiente phi.

No debe usarse la fórmula simplificada de Spearman sin corrección por empates.

La matriz de 12×12 se conserva en el **orden del CSV original** y también se muestra reordenada según las filas ROC del ejercicio.

In [ ]:
import numpy as np
from scipy.stats import spearmanr

A0 = np.asarray(A, dtype=float)
rho = spearmanr(A0, axis=1).statistic
assert rho.shape == (12, 12)
assert np.allclose(np.diag(rho), 1.0)
assert np.allclose(rho, rho.T)

# Verificación independiente mediante phi de la matriz de contingencia 2x2.
def phi(x, y):
    n11 = int(np.sum((x == 1) & (y == 1)))
    n10 = int(np.sum((x == 1) & (y == 0)))
    n01 = int(np.sum((x == 0) & (y == 1)))
    n00 = int(np.sum((x == 0) & (y == 0)))
    return (n11 * n00 - n10 * n01) / np.sqrt((n11+n10)*(n01+n00)*(n11+n01)*(n10+n00))

phi_matrix = np.array([[phi(A0[j], A0[k]) for k in range(12)] for j in range(12)])
assert np.allclose(rho, phi_matrix, atol=1e-12)

sector_roc = [0,2,1,3,10,11,4,5,6,7,8,9]
tema_roc = [9,0,3,11,6,1,4,7,8,5,10,2]
print('Spearman Agroindustria–Energía:', round(float(rho[0,3]), 4))
print('Spearman Minería–Recursos forestales:', round(float(rho[4,5]), 4))
print('Matriz Spearman en orden ROC (redondeada):')
print(np.round(rho[np.ix_(sector_roc, sector_roc)], 2))

## 6. Utilización U ponderada por afinidad

La medida U no es capacidad ni utilización de máquinas: representa la parte del interés de cada sector por una temática que también comparten sus sectores afines.

\[
\rho^+_{jk}=\max(0,\rho_{jk}),\qquad \rho^+_{jj}=1
\]

\[
U_{ji}=a_{ji}\frac{\sum_k\rho^+_{jk}a_{ki}}{\sum_k\rho^+_{jk}}
\]

La suma U debe ser aproximadamente **32,3966204153** (32,40 con dos decimales).

In [ ]:
rho_plus = np.maximum(0, rho)
np.fill_diagonal(rho_plus, 1)
U = A0 * ((rho_plus @ A0) / rho_plus.sum(axis=1)[:, None])
assert np.all((U >= 0) & (U <= 1 + 1e-12))
assert np.isclose(U.sum(), 32.39662041527953, atol=1e-10)
assert np.isclose(U[0,9], 1.0)
assert np.isclose(U[0,0], 0.873630705569315, atol=1e-8)
assert np.isclose(U[0,1], 0.4191195346485304, atol=1e-8)
print('Total U:', format(U.sum(), '.10f'))
print('Matriz U, filas y columnas ROC:')
print(np.round(U[np.ix_(sector_roc,tema_roc)], 2))
print('Suma por sector ROC:', np.round(U[sector_roc].sum(axis=1), 2))
print('Suma por temática ROC:', np.round(U[:, tema_roc].sum(axis=0), 2))

## 7. Heurística: recorrer ROC y elegir el mínimo solapamiento ponderado

Se recorren las columnas en el orden ROC; cada temática se asigna a la franja con menor costo de interés compartido y tres cupos disponibles. El desempate utiliza la primera franja.

\[
c_{is}=\sum_j\min\left(U_{ji},\max_{k\in s}U_{jk}\right)
\]

Comparamos la agrupación ROC original, la solución del modelo con Solver y la solución heurística basada en afinidad.

In [ ]:
def weighted_loss(groups):
    return float(sum(np.sum(U[:, group].sum(axis=1) - U[:, group].max(axis=1)) for group in groups))

def marginal_cost(tema, group):
    if not group:
        return 0.0
    return float(np.minimum(U[:, tema], U[:, group].max(axis=1)).sum())

slots = [[], [], [], []]
steps = []
for it, tema in enumerate(tema_roc, start=1):
    costs = [marginal_cost(tema, slot) if len(slot) < 3 else float('inf') for slot in slots]
    chosen = min(range(4), key=lambda s: costs[s])  # desempate: primera
    slots[chosen].append(tema)
    steps.append((it, tematicas[tema], chosen + 1, costs[chosen]))

expected = [[9,1,2], [0,8,5], [3,6,7], [11,4,10]]
assert slots == expected
assert loss_for_groups(slots) == 7
assert np.isclose(weighted_loss(slots), 3.6180913421533427, atol=1e-10)
print('Iteración | temática | sesión | costo | costo acumulado')
cumulative = 0.0
for it, tema, sesion, marginal in steps:
    cumulative += marginal
    print(f'{it:2} | {tema:34} | S{sesion} | {marginal:.4f} | {cumulative:.4f}')
print('Agenda heurística:', slots)
print('Costo final:', weighted_loss(slots))

## 8. Tercera evaluación y demostración de optimalidad ponderada

El indicador **P** mide interés ponderado perdido y se contrasta con la cota inferior: para cada sector que tiene más de cuatro temáticas de interés, al menos las `d_j-4` menores utilizaciones no podrán ser aprovechadas.

La comparación utiliza el ordenamiento ROC, la solución **exacta** almacenada en la hoja `Optimizacion` del Excel final, y la heurística ROC + U. No modifica los resultados originales de 18 y 7 oportunidades perdidas.

In [ ]:
lower_bound_U = sum(
    sum(sorted(U[j, A0[j] == 1])[:max(int(A0[j].sum()) - 4, 0)])
    for j in range(12)
)
assert np.isclose(lower_bound_U, 3.6180913421533427, atol=1e-10)
assert np.isclose(lower_bound_U, weighted_loss(slots), atol=1e-10)

for label, groups in [
    ('ROC consecutivo', roc_groups),
    ('Optimización con Solver', optimal),
    ('ROC + heurística U', slots),
]:
    L = loss_for_groups(groups)
    P = weighted_loss(groups)
    coverage = (1 - P / U.sum()) * 100
    print(f'{label:25s}: pérdidas = {L:2}, P = {P:.4f}, interés atendido = {coverage:.2f}%')

assert loss_for_groups(roc_groups) == 18
assert loss_for_groups(optimal) == 7
assert np.isclose(weighted_loss(roc_groups), 12.303706253779726, atol=1e-10)
assert np.isclose(weighted_loss(optimal), 3.8032088383465763, atol=1e-10)
print('Cota inferior U:', lower_bound_U)
print('Prueba: la heurística alcanza la cota inferior en pérdidas y en P.')